# 03 — ResNet50 Multi-Label Baseline

**What this notebook does**
1. Trains a ResNet50 image classifier to predict which of the 7 Harvard food groups are present in a plate photo (multi-label binary classification).
2. Evaluates per-group precision, recall, and F1 on the validation set.
3. Compares group-detection accuracy against SegFormer to show why segmentation is needed for health scoring.

**Why this exists:** ResNet50 can detect *which* food groups appear on a plate but has no concept of proportions. It cannot answer "how much" — which means it cannot compute a health score. This notebook exists to demonstrate that limitation as a report baseline.

**Expected runtime:** ~45 min on M2 MPS, ~10 min on Colab T4.

## 1. Config

In [1]:
IMAGE_SIZE    = 224      # ResNet50 standard input
BATCH_SIZE    = 32
NUM_EPOCHS    = 10
LEARNING_RATE = 1e-4
WEIGHT_DECAY  = 1e-4
NUM_WORKERS   = 0        # 0 required in Jupyter on macOS
PRESENCE_THRESHOLD = 0.05  # food group is "present" if it covers >5% of food pixels
PRED_THRESHOLD     = 0.40  # sigmoid output threshold for positive prediction
SEED          = 42

SAVE_PATH    = '../checkpoints/resnet50_multilabel.pt'
HISTORY_PATH = '../checkpoints/resnet50_history.json'

## 2. Environment

In [2]:
import os, sys, json, time, random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image

sys.path.insert(0, os.path.abspath('..'))
from smartplate import load_class_to_group, ALL_GROUPS, mask_to_proportions

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = 'mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Torch : {torch.__version__}')
print(f'Device: {device}')

NUM_GROUPS   = len(ALL_GROUPS)  # 7
class_to_group = load_class_to_group('../assets/foodseg103_to_groups.csv')
GROUP_NAMES  = list(ALL_GROUPS)
print(f'Groups: {GROUP_NAMES}')

Torch : 2.7.1
Device: mps
Groups: ['vegetables', 'fruits', 'whole_grains', 'healthy_protein', 'refined_grains', 'red_processed_meat', 'sugary_fatty']


## 3. Dataset

Labels are derived from the ground-truth segmentation masks: a food group is marked **present** (1) if it covers more than `PRESENCE_THRESHOLD` of the food pixels in the image.

In [3]:
from datasets import load_dataset

raw = load_dataset('EduardoPacheco/FoodSeg103')
print(raw)

def make_label(mask_pil):
    mask = np.array(mask_pil)
    props = mask_to_proportions(mask, class_to_group)
    return torch.tensor([float(props[g] > PRESENCE_THRESHOLD) for g in GROUP_NAMES], dtype=torch.float32)

train_tf = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_tf = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class FoodGroupDataset(Dataset):
    def __init__(self, hf_split, transform):
        self.data      = hf_split
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        ex    = self.data[idx]
        image = self.transform(ex['image'].convert('RGB'))
        label = make_label(ex['label'])
        return image, label

train_ds = FoodGroupDataset(raw['train'],      train_tf)
val_ds   = FoodGroupDataset(raw['validation'], val_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
print(f'Train: {len(train_ds)} samples  |  Val: {len(val_ds)} samples')

# Label distribution
pos = torch.zeros(NUM_GROUPS)
for _, lbl in train_loader:
    pos += lbl.sum(dim=0)
print('\nTrain label counts (positive samples per group):')
for g, c in zip(GROUP_NAMES, pos.int().tolist()):
    print(f'  {g:22s}  {c}')

/Users/krishdembla/miniconda3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['image', 'label', 'classes_on_image', 'id'],
        num_rows: 4983
    })
    validation: Dataset({
        features: ['image', 'label', 'classes_on_image', 'id'],
        num_rows: 2135
    })
})
Train: 4983 samples  |  Val: 2135 samples

Train label counts (positive samples per group):
  vegetables              3102
  fruits                  1558
  whole_grains            346
  healthy_protein         1570
  refined_grains          2416
  red_processed_meat      1410
  sugary_fatty            1947


## 4. Model

In [4]:
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
model.fc = nn.Linear(model.fc.in_features, NUM_GROUPS)
model = model.to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
print(f'ResNet50 trainable params: {n_params:.1f} M')
print(f'Output: {NUM_GROUPS} food groups (sigmoid multi-label)')

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /Users/krishdembla/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:09<00:00, 11.1MB/s]


ResNet50 trainable params: 23.5 M
Output: 7 food groups (sigmoid multi-label)


## 5. Training

In [5]:
from tqdm.auto import tqdm

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

def evaluate(model, loader):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            total_loss += criterion(logits, labels).item()
            all_preds.append((logits.sigmoid() > PRED_THRESHOLD).cpu().float())
            all_labels.append(labels.cpu())
    preds  = torch.cat(all_preds)
    labels = torch.cat(all_labels)
    tp = (preds * labels).sum(0)
    fp = (preds * (1 - labels)).sum(0)
    fn = ((1 - preds) * labels).sum(0)
    precision = tp / (tp + fp + 1e-8)
    recall    = tp / (tp + fn + 1e-8)
    f1        = 2 * precision * recall / (precision + recall + 1e-8)
    return total_loss / len(loader), precision, recall, f1

Path(SAVE_PATH).parent.mkdir(parents=True, exist_ok=True)
history, best_f1 = [], -1.0

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    t0 = time.time()
    running_loss, n = 0.0, 0
    for images, labels in tqdm(train_loader, desc=f'epoch {epoch}/{NUM_EPOCHS}', leave=False):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item(); n += 1
    scheduler.step()

    val_loss, prec, rec, f1 = evaluate(model, val_loader)
    mean_f1 = f1.mean().item()
    dt = time.time() - t0
    print(f'epoch {epoch:2d}: train_loss={running_loss/n:.3f}  val_loss={val_loss:.3f}  mean_F1={mean_f1:.3f}  ({dt/60:.1f} min)')

    history.append({'epoch': epoch, 'train_loss': running_loss/n, 'val_loss': val_loss, 'mean_f1': mean_f1})
    if mean_f1 > best_f1:
        best_f1 = mean_f1
        torch.save(model.state_dict(), SAVE_PATH)
        print(f'  ↳ saved best (mean_F1={best_f1:.3f})')

with open(HISTORY_PATH, 'w') as f:
    json.dump(history, f, indent=2)
print('\nTraining complete.')

epoch  1: train_loss=0.494  val_loss=0.396  mean_F1=0.727  (4.5 min)
  ↳ saved best (mean_F1=0.727)


epoch  2: train_loss=0.335  val_loss=0.349  mean_F1=0.768  (4.4 min)
  ↳ saved best (mean_F1=0.768)


epoch  3: train_loss=0.249  val_loss=0.347  mean_F1=0.772  (4.5 min)
  ↳ saved best (mean_F1=0.772)


epoch  4: train_loss=0.178  val_loss=0.363  mean_F1=0.779  (4.6 min)
  ↳ saved best (mean_F1=0.779)


epoch  5: train_loss=0.124  val_loss=0.395  mean_F1=0.776  (4.9 min)


epoch  6: train_loss=0.087  val_loss=0.399  mean_F1=0.784  (5.2 min)
  ↳ saved best (mean_F1=0.784)


epoch  7: train_loss=0.059  val_loss=0.406  mean_F1=0.782  (4.8 min)


epoch  8: train_loss=0.046  val_loss=0.421  mean_F1=0.776  (5.0 min)


epoch  9: train_loss=0.041  val_loss=0.425  mean_F1=0.780  (4.8 min)


epoch 10: train_loss=0.036  val_loss=0.422  mean_F1=0.779  (5.2 min)

Training complete.


## 6. Per-group evaluation

In [6]:
import pandas as pd

model.load_state_dict(torch.load(SAVE_PATH, map_location=device))
_, prec, rec, f1 = evaluate(model, val_loader)

results = pd.DataFrame({
    'food_group': GROUP_NAMES,
    'precision':  prec.numpy().round(3),
    'recall':     rec.numpy().round(3),
    'f1':         f1.numpy().round(3),
})
print(results.to_string(index=False))
print(f'\nMean F1: {f1.mean():.3f}  |  Mean precision: {prec.mean():.3f}  |  Mean recall: {rec.mean():.3f}')
results.to_csv('../figures/resnet50_group_f1.csv', index=False)

        food_group  precision  recall    f1
        vegetables      0.904   0.914 0.909
            fruits      0.833   0.847 0.840
      whole_grains      0.941   0.864 0.901
   healthy_protein      0.664   0.673 0.669
    refined_grains      0.755   0.799 0.776
red_processed_meat      0.719   0.704 0.711
      sugary_fatty      0.682   0.677 0.679

Mean F1: 0.784  |  Mean precision: 0.785  |  Mean recall: 0.783


## 7. Why ResNet50 cannot health-score a plate

ResNet50 tells you **which** food groups are on a plate (yes/no). A health score needs **how much** of each group is present. This cell shows the gap on one real example.

In [9]:
# Pick a validation example with multiple food groups present
idx = next(i for i in range(len(raw['validation']))
           if sum(mask_to_proportions(np.array(raw['validation'][i]['label']), class_to_group).values()) > 0.5
           and sum(v > PRESENCE_THRESHOLD
                   for v in mask_to_proportions(np.array(raw['validation'][i]['label']), class_to_group).values()) >= 3)

ex = raw['validation'][idx]
mask = np.array(ex['label'])
true_props = mask_to_proportions(mask, class_to_group)

# ResNet50 prediction
model.eval()
img_tensor = val_tf(ex['image'].convert('RGB')).unsqueeze(0).to(device)
with torch.no_grad():
    probs = model(img_tensor).sigmoid().squeeze().cpu()
resnet_preds = {g: bool((probs[i] > PRED_THRESHOLD).item()) for i, g in enumerate(GROUP_NAMES)}

print("Example plate — what each model tells you:\n")
print(f"{'Food group':<22}  {'ResNet50':>12}  {'SegFormer (proportion)':>22}")
print("-" * 60)
for g in GROUP_NAMES:
    present = "present" if resnet_preds[g] else "absent"
    pct = true_props[g] * 100
    seg_str = f"{pct:.1f}%" if pct > 0.1 else "—"
    print(f"{g:<22}  {present:>12}  {seg_str:>22}")

print()

Example plate — what each model tells you:

Food group                  ResNet50  SegFormer (proportion)
------------------------------------------------------------
vegetables                   present                   68.8%
fruits                        absent                    5.4%
whole_grains                  absent                       —
healthy_protein              present                    7.8%
refined_grains               present                   18.0%
red_processed_meat            absent                       —
sugary_fatty                  absent                       —



Here you can see that ResNet50 will show you which groups are present, but not how much of each. So a plate with 70% fried chicken and 20% refined grains and 10% vegetables may seem like a healthier plate just by ResNet50 classification since you see vegetables in it. But the fact that most of it is fried junk foods, makes it unhealthy and that is something that can be ascertained only through segmentation masking.